<a href="https://colab.research.google.com/github/CienciaDatosUdea/002_EstudiantesAprendizajeEstadistico/blob/main/semestre2025-2/Sesiones/Sesion_06_normal_equation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 6: Regresión lineal multivariada y ecuación normal

**Objetivos**
1. Escribir la regresión lineal en forma matricial.
2. Derivar la ecuación normal y entender cuándo tiene solución única.
3. Resolverla numéricamente de forma estable.
4. Interpretar mínimos cuadrados como máxima verosimilitud (MLE) y la regularización L2 como máximo a posteriori (MAP).

## 1. Introducción: regresión lineal multivariada en forma matricial

En la sesión de regresión lineal simple predijimos una cantidad $y$ a partir de una sola variable $x$. En un experimento real casi siempre medimos varias cantidades a la vez. Por ejemplo, el precio de una vivienda depende de su área, del número de habitaciones y de su ubicación. En general tendremos $n$ **características** $x_1, x_2, \dots, x_n$, medidas en $m$ **muestras**, y queremos un modelo lineal que prediga $y$ a partir de ellas.

### Los datos como una tabla

Organizamos los datos como en una bitácora de laboratorio o un `DataFrame`: **cada fila es una muestra y cada columna es una característica**.

| Muestra | $x_1$ | $x_2$ | $\cdots$ | $x_n$ | $y$ |
|---|---|---|---|---|---|
| 1 | $x_1^{(1)}$ | $x_2^{(1)}$ | $\cdots$ | $x_n^{(1)}$ | $y^{(1)}$ |
| 2 | $x_1^{(2)}$ | $x_2^{(2)}$ | $\cdots$ | $x_n^{(2)}$ | $y^{(2)}$ |
| $\vdots$ | $\vdots$ | $\vdots$ | | $\vdots$ | $\vdots$ |
| $m$ | $x_1^{(m)}$ | $x_2^{(m)}$ | $\cdots$ | $x_n^{(m)}$ | $y^{(m)}$ |

La notación es la siguiente:

- El superíndice entre paréntesis $(i)$ indica la **muestra**, con $i = 1, \dots, m$. No es una potencia.
- El subíndice $j$ indica la **característica**, con $j = 1, \dots, n$.
- $x_j^{(i)}$ es el valor de la característica $j$ en la muestra $i$.
- $y^{(i)}$ es el valor **observado** (la etiqueta) de la muestra $i$.
- $\hat{y}^{(i)}$ es el valor **predicho** por el modelo para esa muestra.

### El modelo para una muestra

La hipótesis lineal es

$$
\hat{y}^{(i)} = h_\theta(x^{(i)}) = \theta_0 + \theta_1 x_1^{(i)} + \theta_2 x_2^{(i)} + \cdots + \theta_n x_n^{(i)} .
$$

Para escribirla de forma compacta, añadimos una característica ficticia $x_0^{(i)} = 1$ que acompaña al intercepto $\theta_0$. Así, cada muestra y los parámetros se representan como **vectores columna** de dimensión $n+1$:

$$
x^{(i)} = \begin{bmatrix} 1 \\ x_1^{(i)} \\ \vdots \\ x_n^{(i)} \end{bmatrix},
\qquad
\theta = \begin{bmatrix} \theta_0 \\ \theta_1 \\ \vdots \\ \theta_n \end{bmatrix},
\qquad
h_\theta(x^{(i)}) = \theta^T x^{(i)} = \left(x^{(i)}\right)^T \theta .
$$

(En el resto del notebook, $x^{(i)}$ siempre incluye este 1.)

### El modelo para todas las muestras: $\hat{y} = X\theta$

Apilamos las $m$ muestras como **filas** de una matriz, la **matriz de diseño**. Su primera columna es de unos, y es exactamente la tabla anterior sin la columna $y$:

$$
X =
\begin{bmatrix}
\left(x^{(1)}\right)^T \\
\left(x^{(2)}\right)^T \\
\vdots \\
\left(x^{(m)}\right)^T
\end{bmatrix}
=
\begin{bmatrix}
1 & x_1^{(1)} & x_2^{(1)} & \cdots & x_n^{(1)} \\
1 & x_1^{(2)} & x_2^{(2)} & \cdots & x_n^{(2)} \\
\vdots & \vdots & \vdots & & \vdots \\
1 & x_1^{(m)} & x_2^{(m)} & \cdots & x_n^{(m)}
\end{bmatrix}_{m \times (n+1)},
\qquad
y =
\begin{bmatrix} y^{(1)} \\ y^{(2)} \\ \vdots \\ y^{(m)} \end{bmatrix}_{m \times 1}.
$$

El producto matriz–vector calcula las $m$ predicciones de una sola vez. La fila $i$ de $X\theta$ es $\left(x^{(i)}\right)^T\theta = \hat{y}^{(i)}$:

$$
\hat{y} = X\theta,
\qquad
\underbrace{(m \times (n+1))}_{X}\;\underbrace{((n+1) \times 1)}_{\theta} = \underbrace{(m \times 1)}_{\hat{y}} .
$$

Esta es la convención que usa el código. `df[["ones", "X1", ...]].values` produce justamente una matriz $m \times (n+1)$, y las predicciones se calculan como `X @ theta`. Es también la convención de CS229, Bishop y scikit-learn.

**Interpretación geométrica.** Con una característica ($n = 1$), $h_\theta$ es la recta de la sesión anterior. Con dos características es un plano en el espacio $(x_1, x_2, y)$. En general es un hiperplano de dimensión $n$ en $\mathbb{R}^{n+1}$.

### Función de coste

Medimos qué tan bien se ajusta el modelo con el vector de **residuos** $X\theta - y \in \mathbb{R}^m$, cuya componente $i$ es $\hat{y}^{(i)} - y^{(i)}$. La función de coste es el error cuadrático medio (con un factor $\tfrac{1}{2}$ por conveniencia al derivar):

$$
J(\theta_0, \theta_1, \dots, \theta_n)
= \frac{1}{2m} \sum_{i=1}^{m} \left( \theta^T x^{(i)} - y^{(i)} \right)^2
= \frac{1}{2m} \, \lVert X\theta - y \rVert_2^2
= \frac{1}{2m} \, (X\theta - y)^T (X\theta - y) .
$$

Geométricamente, $J$ es proporcional al cuadrado de la norma euclidiana del vector de residuos. Es decir, suma los cuadrados de las distancias **verticales** entre cada dato y el hiperplano. Existen otras funciones de coste: [lista](https://jmlb.github.io/flashcards/2018/04/21/list_cost_functions_fo_neuralnets/).

**Objetivo:** encontrar el vector $\theta$ que minimiza $J(\theta)$. En sesiones anteriores lo hicimos iterativamente con descenso del gradiente; en la siguiente sección obtendremos la solución exacta.

## 2. Ecuación normal

En lugar de descenso del gradiente (sesiones anteriores), podemos encontrar el mínimo de $J$ de forma exacta, igualando su gradiente a cero.

**Tarea.** Usando las propiedades (con $a, b, z, \theta$ vectores y $A$ matriz simétrica)

- $z^T z = \sum_i z_i^2$
- $a^T b = b^T a$ (es un escalar)
- $\nabla_\theta\, b^T\theta = b$
- $\nabla_\theta\, \theta^T A\theta = 2A\theta$  (en general, $(A + A^T)\theta$)

demuestre que

1. $J(\theta) = \dfrac{1}{2m}(X\theta - y)^T(X\theta - y)$
2. $\nabla_\theta J = \dfrac{1}{m}\left(X^T X\theta - X^T y\right)$
3. Si $X^TX$ es invertible, $\nabla_\theta J = 0 \Rightarrow \boxed{\theta = (X^TX)^{-1}X^Ty}$

### Demostración

Con $X \in \mathbb{R}^{m\times(n+1)}$, $\theta \in \mathbb{R}^{(n+1)\times 1}$, $y \in \mathbb{R}^{m\times 1}$:

$$
2m\,J = (X\theta - y)^T(X\theta - y) = \theta^T X^T X\theta - \theta^T X^T y - y^T X\theta + y^T y .
$$

Como $y^T X\theta$ es un escalar, $y^T X\theta = (y^T X\theta)^T = \theta^T X^T y$, así que

$$
2m\,J = \theta^T (X^T X)\theta - 2\,\theta^T (X^T y) + y^T y .
$$

$A = X^TX$ es simétrica y $b = X^Ty$. Aplicando las propiedades:

$$
2m\,\nabla_\theta J = 2X^TX\theta - 2X^Ty
\quad\Rightarrow\quad
\nabla_\theta J = \frac{1}{m}\left(X^TX\theta - X^Ty\right).
$$

Igualando a cero se obtienen las **ecuaciones normales** $X^TX\theta = X^Ty$. Multiplicando **por la izquierda** por $(X^TX)^{-1}$:

$$
(X^TX)^{-1}(X^TX)\,\theta = (X^TX)^{-1}X^Ty \quad\Rightarrow\quad \theta = (X^TX)^{-1}X^Ty .
$$

**Observaciones**

- **Es un mínimo:** el hessiano $\nabla^2_\theta J = \frac{1}{m}X^TX$ es semidefinido positivo, así que $J$ es convexa.
- **Existencia de la inversa:** $X^TX$ es invertible si y solo si las columnas de $X$ son linealmente independientes (requiere $m \ge n+1$ y ninguna característica colineal con otras). Si no, se usa la pseudoinversa o regularización (sección 6).
- **Interpretación geométrica:** $X\theta$ es la proyección ortogonal de $y$ sobre el espacio columna de $X$; el residuo cumple $X^T(y - X\theta) = 0$.
- **Costo computacional:** resolver el sistema cuesta $O(n^3)$; descenso del gradiente es preferible cuando $n$ es muy grande (del orden de $10^4$ o más).
- **Numéricamente** no se calcula la inversa explícita: se usa `np.linalg.solve` o `np.linalg.lstsq`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # semilla para reproducibilidad

In [ ]:
def normal_equation(X, y):
    '''Resuelve X^T X theta = X^T y sin invertir la matriz explícitamente.'''
    return np.linalg.solve(X.T @ X, X.T @ y)

def add_bias(*features):
    '''Construye la matriz de diseño m x (n+1) con una columna de unos.'''
    m = len(features[0])
    return np.column_stack([np.ones(m), *features])

## 3. Regresión lineal simple (datos sintéticos)

Generamos datos con parámetros conocidos **y con ruido**, para comprobar que el método los recupera. Sin ruido el ajuste es trivial y exacto.

In [ ]:
N = 30
theta_true = np.array([1.0, 2.0])      # [intercepto, pendiente]
sigma = 0.3

x1 = np.linspace(-1, 1, N)
y = theta_true[0] + theta_true[1]*x1 + rng.normal(0, sigma, N)

df = pd.DataFrame({"Y": y, "X1": x1})
df.head()

In [ ]:
X = add_bias(df["X1"].to_numpy())       # (N, 2)
y = df["Y"].to_numpy()                   # (N,)
print("X:", X.shape, " y:", y.shape)

theta = normal_equation(X, y)
print("theta estimado:", theta)
print("theta real:    ", theta_true)

# Verificación con dos métodos independientes
theta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("lstsq:         ", theta_lstsq)

In [ ]:
x_ = np.linspace(-1, 1, 100)
plt.plot(df.X1, df.Y, "ro", label="datos")
plt.plot(x_, theta[0] + theta[1]*x_, "b-", label="ajuste (ecuación normal)")
plt.plot(x_, theta_true[0] + theta_true[1]*x_, "k--", label="modelo real")
plt.xlabel("$x_1$"); plt.ylabel("$y$"); plt.legend();

### Incertidumbre de los parámetros

Para ciencias es esencial reportar barras de error. Si el ruido es gaussiano i.i.d. con varianza $\sigma^2$,

$$
\operatorname{Cov}(\hat\theta) = \sigma^2 (X^TX)^{-1},
\qquad
\hat\sigma^2 = \frac{1}{m-(n+1)}\lVert y - X\hat\theta\rVert^2 .
$$

In [ ]:
m, p = X.shape
residuals = y - X @ theta
sigma2_hat = residuals @ residuals / (m - p)
cov_theta = sigma2_hat * np.linalg.inv(X.T @ X)
for name, t, s in zip(["theta_0", "theta_1"], theta, np.sqrt(np.diag(cov_theta))):
    print(f"{name} = {t:.3f} ± {s:.3f}")
print(f"sigma estimado = {np.sqrt(sigma2_hat):.3f}  (real = {sigma})")

## 4. Modelo con dos características

Plano real: $y = -1.0 + 0.2\,x_1 + 0.5\,x_2$.

In [ ]:
theta_true2 = np.array([-1.0, 0.2, 0.5])

g = np.linspace(-1, 1, 50)
G1, G2 = np.meshgrid(g, g)
G_Y = theta_true2[0] + theta_true2[1]*G1 + theta_true2[2]*G2

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_surface(G1, G2, G_Y, alpha=0.6)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_zlabel("$y$");

In [ ]:
# Muestreamos puntos aleatorios en [-1, 1]^2 y añadimos ruido
N = 200
x1 = rng.uniform(-1, 1, N)
x2 = rng.uniform(-1, 1, N)
y = theta_true2[0] + theta_true2[1]*x1 + theta_true2[2]*x2 + rng.normal(0, 0.1, N)

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_surface(G1, G2, G_Y, alpha=0.2)
ax.scatter(x1, x2, y, color="green", s=8)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_zlabel("$y$");

In [ ]:
df = pd.DataFrame({"Y": y, "X1": x1, "X2": x2})
X = add_bias(df["X1"].to_numpy(), df["X2"].to_numpy())   # (N, 3)
y = df["Y"].to_numpy()

theta = normal_equation(X, y)
print("theta estimado:", theta)
print("theta real:    ", theta_true2)

from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(df[["X1", "X2"]], df["Y"])
print("sklearn:       ", np.r_[reg.intercept_, reg.coef_])

### Ejercicio: colinealidad

Haga `x2 = 2*x1` y vuelva a resolver. ¿Qué pasa con `np.linalg.solve`? Compare con `np.linalg.pinv(X) @ y` y con `np.linalg.cond(X.T @ X)`.

## 5. Datos reales: vivienda en California

> El conjunto *Boston housing* fue retirado de scikit-learn (v1.2) por problemas éticos en una de sus variables. Usamos *California housing*.

Ajustamos el valor mediano de las viviendas (`MedHouseVal`, en cientos de miles de USD) en función del ingreso mediano del bloque (`MedInc`, en decenas de miles de USD).

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
df = housing.frame[["MedInc", "MedHouseVal"]]
df.describe()

In [ ]:
plt.plot(df.MedInc, df.MedHouseVal, "go", alpha=0.1, ms=2)
plt.xlabel("MedInc (x 10k USD)"); plt.ylabel("MedHouseVal (x 100k USD)");
# Observe el tope en 5.0: los valores están censurados, lo cual sesga el ajuste.

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=0)

X_train = add_bias(df_train["MedInc"].to_numpy())
y_train = df_train["MedHouseVal"].to_numpy()
X_test  = add_bias(df_test["MedInc"].to_numpy())
y_test  = df_test["MedHouseVal"].to_numpy()

theta = normal_equation(X_train, y_train)
print("theta:", theta)

def r2(y, y_pred):
    return 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)

for name, Xs, ys in [("train", X_train, y_train), ("test", X_test, y_test)]:
    y_pred = Xs @ theta
    print(f"{name}: MSE = {np.mean((ys - y_pred)**2):.3f}, R^2 = {r2(ys, y_pred):.3f}")

In [ ]:
x = np.linspace(0, 15, 100)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df_train.MedInc, df_train.MedHouseVal, "go", alpha=0.1, ms=2)
axes[0].plot(x, theta[0] + theta[1]*x, "b-")
axes[0].set_xlabel("MedInc"); axes[0].set_ylabel("MedHouseVal")

res = y_test - X_test @ theta
axes[1].plot(X_test[:, 1], res, "ko", alpha=0.1, ms=2)
axes[1].axhline(0, color="r")
axes[1].set_xlabel("MedInc"); axes[1].set_ylabel("residuo")
plt.tight_layout();

## 6. Interpretación probabilística: mínimos cuadrados = máxima verosimilitud

Supongamos que cada observación es

$$
y^{(i)} = \theta^T x^{(i)} + \epsilon^{(i)},
$$

donde los errores $\epsilon^{(i)}$ son **independientes e idénticamente distribuidos (i.i.d.)**, gaussianos con media cero y **varianza $\sigma^2$**:

$$
p(\epsilon^{(i)}) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(\epsilon^{(i)})^2}{2\sigma^2}\right).
$$

(Aquí $x^{(i)}$ incluye el 1 del intercepto.) Equivalentemente,

$$
p(y^{(i)}\mid x^{(i)};\theta) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(y^{(i)} - \theta^T x^{(i)})^2}{2\sigma^2}\right).
$$

Por independencia, la **verosimilitud** de las $m$ muestras es

$$
\mathcal{L}(\theta) = p(y\mid X;\theta) = \prod_{i=1}^{m} p(y^{(i)}\mid x^{(i)};\theta).
$$

El estimador de máxima verosimilitud elige el $\theta$ que maximiza $\mathcal{L}$. Como el logaritmo es monótono creciente, maximizamos

$$
\ell(\theta) = \ln\mathcal{L}(\theta) = m\ln\frac{1}{\sqrt{2\pi\sigma^2}} - \frac{1}{2\sigma^2}\sum_{i=1}^{m}\left(y^{(i)} - \theta^T x^{(i)}\right)^2 .
$$

El primer término no depende de $\theta$, así que

$$
\arg\max_\theta \ell(\theta) = \arg\min_\theta \sum_{i=1}^{m}\left(y^{(i)} - \theta^T x^{(i)}\right)^2 = \arg\min_\theta J(\theta).
$$

**Conclusión:** bajo ruido gaussiano i.i.d., mínimos cuadrados es el estimador de máxima verosimilitud, y el resultado no depende de $\sigma$. Si cada punto tiene su propia incertidumbre $\sigma_i$ (como en el laboratorio), se obtiene mínimos cuadrados ponderados, es decir, la minimización de $\chi^2 = \sum_i (y^{(i)} - \theta^T x^{(i)})^2/\sigma_i^2$.

## 7. Interpretación probabilística de la regularización (MAP)

Ahora tratamos $\theta$ como variable aleatoria con una distribución *a priori* gaussiana isotrópica de varianza $\tau^2$ (con $d = n+1$ componentes):

$$
p(\theta) = \frac{1}{(2\pi\tau^2)^{d/2}}\exp\!\left(-\frac{\lVert\theta\rVert^2}{2\tau^2}\right).
$$

Las $x^{(i)}$ se consideran dadas (no modelamos su distribución). Por el teorema de Bayes:

$$
p(\theta\mid X, y) = \frac{p(y\mid X,\theta)\,p(\theta)}{p(y\mid X)},
$$

donde usamos que $p(\theta\mid X) = p(\theta)$ ($X$ no aporta información sobre $\theta$ por sí sola). El denominador no depende de $\theta$, así que el estimador **máximo a posteriori (MAP)** maximiza el numerador:

$$
\theta_{MAP} = \arg\max_\theta \left[\sum_{i=1}^m \ln p(y^{(i)}\mid x^{(i)},\theta) + \ln p(\theta)\right]
= \arg\min_\theta \left[\frac{1}{2\sigma^2}\sum_{i=1}^m\left(y^{(i)} - \theta^Tx^{(i)}\right)^2 + \frac{1}{2\tau^2}\lVert\theta\rVert^2\right].
$$

Multiplicando por $\sigma^2/m$ (no cambia el mínimo) y usando $J = \frac{1}{2m}\sum(\cdot)^2$:

$$
\theta_{MAP} = \arg\min_\theta \left[J(\theta) + \lambda\lVert\theta\rVert^2\right],
\qquad \lambda = \frac{\sigma^2}{2m\,\tau^2}.
$$

Esto es la **regresión ridge** (regularización L2 o de Tikhonov). Su ecuación normal es

$$
\theta_{ridge} = \left(X^TX + \frac{\sigma^2}{\tau^2} I\right)^{-1} X^T y,
$$

que siempre es invertible para $\tau$ finito, aun si $X^TX$ es singular. En la práctica el intercepto $\theta_0$ normalmente no se regulariza y las características se estandarizan antes.

- Cuando $\tau\to\infty$ (a priori plano, poco informativo) se recupera mínimos cuadrados.
- Cuando $\tau$ es pequeño, el término de regularización domina y los parámetros se **encogen** hacia cero.

**Importante:** la regularización L2 reduce la magnitud de los parámetros, pero en general **no los hace exactamente cero**, así que no reduce el número de parámetros. Los coeficientes exactamente nulos (selección de variables) aparecen con un a priori de Laplace, $p(\theta)\propto e^{-\lVert\theta\rVert_1/b}$, que da la regularización L1 (**Lasso**). Ambas reducen la complejidad efectiva del modelo (menor varianza, mayor sesgo).

In [ ]:
# Ejemplo: ridge con la ecuación normal (sin regularizar el intercepto)
def ridge_normal_equation(X, y, alpha):
    I = np.eye(X.shape[1])
    I[0, 0] = 0.0                     # no penalizar theta_0
    return np.linalg.solve(X.T @ X + alpha * I, X.T @ y)

for alpha in [0, 1, 100, 1e4]:
    print(f"alpha = {alpha:>7}: theta = {ridge_normal_equation(X_train, y_train, alpha)}")

### Ejercicios

1. Demuestre el paso de $\ell(\theta)$ a partir del producto de gaussianas.
2. Derive la ecuación normal de ridge a partir de $\nabla_\theta\left[J + \lambda\lVert\theta\rVert^2\right] = 0$.
3. Use todas las características de California housing, estandarícelas y grafique los coeficientes de ridge y lasso (`sklearn.linear_model`) en función de la regularización. ¿Cuál anula coeficientes?

**Referencias:** Ng, *CS229 Lecture Notes*, parte I; Bishop, *PRML*, §3.1–3.3; Deisenroth et al., *Mathematics for Machine Learning*, cap. 9; James et al., *ISLR*, cap. 3 y 6; Mehta et al., *A high-bias, low-variance introduction to Machine Learning for physicists*, Phys. Rep. 810 (2019), §6.